# Model failure selection

On top of the classifier-based selection methods, I also examine the base model's own responses and select pairs where it produced the answer requested by the injection.

The point of this method is to use examples of observed model failures as training data, then compare their fine-tuning results with the other selection methods.

- [Experiment settings](#Experiment-settings)
- [How model failures select the pairs](#How-model-failures-select-the-pairs)
- [Examples of the selected pairs](#Examples-of-the-selected-pairs)


## Experiment settings

I use the same Qwen2.5-7B experiment and FIPI candidate pool for this comparison, keeping the model and number of selected pairs consistent across all methods.

The function `configuration()` reads these settings into the dictionary variable `config`, while the variable `output` identifies the completed experiment used by the result-reading functions.


In [ ]:
# inspect the pairs selected because the base model produced the injected answer

from pathlib import Path
import sys
import json
import pandas as pd

root = next(path for path in [Path.cwd(), *Path.cwd().parents] if (path / "scripts/curation/configs/config_qwen7b.json").exists())
sys.path.insert(0, str(root))

# use the same Qwen2.5-7B configuration across all selection methods
from scripts.curation.selection.extract_results import configuration, selection, validation
config, output = configuration()
pd.set_option("display.max_colwidth", None)

## How model failures select the pairs

I use the responses where the answer check identifies an injection success, and randomly order the corresponding prompt pairs to select 256 of such pairs.

Specifically, the check looks for the injection's requested answer in the injection response and checks that it is absent from the safe response, with the column `witness_redirection` recording this result. The function `selection()` reads the selected pairs, while the column `selection_score` adds a small random number to the answer check to order ties.


In [ ]:
# compare the 256 pairs used to train this method's adapter
selected = selection("model_failures")
# the counts show which injection methods appear in this training group
display(selected.groupby("attack_method").size().rename("selected pairs").to_frame())
display(selected[["pair_id", "selection_score", "witness_redirection"]].head(10))

## Examples of the selected pairs

The table below shows three selected pairs, each containing the original user instruction and both the safe and injection versions of its content. The source column `witness` gives the answer requested by the injection, so the inserted instruction can be compared with what it asks the model to produce.


In [ ]:
# keep the user instruction beside both prompt versions to inspect the inserted injection
display(selected[["user_instruction", "clean_data", "injected_data", "witness"]].head(3))